1. prendere i dati alla vecchia maniera con api
2. prendere tutti i dati dei trade dalla prima data all'ultima rollando limite 1000 per richiesta
3. confrontare dati OG con trades
4. ripetere per ogni candidato della scommessa (che in realtà è il punto 0. perchè da lì prendiamo tutte le condition_id)


5. Nicola voleva fare un state-space model, tipo HMM. 

https://polymarket.com/event/who-will-be-the-next-pope

In [11]:
import requests

# Fetch the event by slug
url = "https://gamma-api.polymarket.com/events"
params = {"slug": "who-will-be-the-next-pope"}

response = requests.get(url, params=params)
events = response.json()

for event in events:
    print(f"Event: {event['title']}")
    for market in event.get("markets", []):
        print(f"  Question: {market['question']}")
        print(f"  Condition ID: {market['conditionId']}")
        print(f"  Question ID: {market.get('questionId', 'N/A')}")
        tokens = market.get("tokens", [])
        for token in tokens:
            print(f"    Token ID: {token['token_id']}, Outcome: {token['outcome']}")
        print()

Event: Who will be the next Pope?
  Question: Will Robert Francis Prevost be the next pope?
  Condition ID: 0x14197a2979391a7bb1e6918582334640f309c22c7718f824c9fa1e149302662f
  Question ID: N/A

  Question: Will Person M be the next pope?
  Condition ID: 0x1300e1cf7068c5581642cedbf18aa37b9089f94a4a732ab918900dd57c6f6033
  Question ID: N/A

  Question: Will another person be the next Pope?
  Condition ID: 0x90dd69cd3383c1a4de4255e2848e367fff87052c1cdb14a575faea068cb8ec06
  Question ID: N/A

  Question: Will Luis Antonio Tagle be the next pope?
  Condition ID: 0x31ffd9aab0ebb5be9ec63c29f42bbe1d6c7a43c246612ce3e42febc8658e242a
  Question ID: N/A

  Question: Will  Anders Arborelius be the next pope?
  Condition ID: 0x3ab814a91628c1ba02720670c334d3d25a9414adca1e74928c6db3f077257219
  Question ID: N/A

  Question: Will Jose Tolentino de Mendonca be the next pope?
  Condition ID: 0x201d65793c52d5fb5a2b992813f1077a3a95f0c6daa676965ac1b49e7182eb1e
  Question ID: N/A

  Question: Will Kurt Koch

In [37]:
import requests
import pandas as pd

url = "https://gamma-api.polymarket.com/events"
params = {"slug": "who-will-be-the-next-pope"}

response = requests.get(url, params=params)
events = response.json()

rows = []
for event in events:
    event_title = event["title"]
    for market in event.get("markets", []):
        question = market["question"]
        condition_id = market["conditionId"]
        question_id = market.get("questionId", "N/A")
        tokens = market.get("tokens", [])
        if tokens:
            for token in tokens:
                rows.append({
                    "event_title": event_title,
                    "question": question,
                    "condition_id": condition_id,
                    "question_id": question_id,
                    "token_id": token["token_id"],
                    "outcome": token["outcome"],
                })
        else:
            rows.append({
                "event_title": event_title,
                "question": question,
                "condition_id": condition_id,
                "question_id": question_id,
                "token_id": None,
                "outcome": None,
            })

df = pd.DataFrame(rows)
df.head()
# df.to_excel("../data/polymarket_pope_event.xlsx", index=False)

,event_title,question,condition_id,question_id,token_id,outcome
0,Who will be the next Pope?,Will Robert Francis Prevost be the next pope?,0x14197a2979391a7bb1e6918582334640f309c22c7718...,N/A,None,None
1,Who will be the next Pope?,Will Person M be the next pope?,0x1300e1cf7068c5581642cedbf18aa37b9089f94a4a73...,N/A,None,None
2,Who will be the next Pope?,Will another person be the next Pope?,0x90dd69cd3383c1a4de4255e2848e367fff87052c1cdb...,N/A,None,None
3,Who will be the next Pope?,Will Luis Antonio Tagle be the next pope?,0x31ffd9aab0ebb5be9ec63c29f42bbe1d6c7a43c24661...,N/A,None,None
4,Who will be the next Pope?,Will Anders Arborelius be the next pope?,0x3ab814a91628c1ba02720670c334d3d25a9414adca1e...,N/A,None,None


In [44]:
ex_cond_id = df['condition_id'].iloc[1]

ex_cond_id

'0x1300e1cf7068c5581642cedbf18aa37b9089f94a4a732ab918900dd57c6f6033'

In [45]:
import requests

def get_prices_history(condition_id, interval="1m", fidelity=1):
    # Step 1: condition_id → token_ids
    markets_url = "https://gamma-api.polymarket.com/markets"
    r = requests.get(markets_url, params={"condition_id": condition_id})
    market = r.json()[0]
    tokens = market.get("tokens", [])

    # Step 2: fetch price history per token
    all_dfs = []
    for token in tokens:
        r = requests.get(
            "https://clob.polymarket.com/prices-history",
            params={"market": token["token_id"], "interval": interval, "fidelity": fidelity}
        )
        history = r.json().get("history", [])
        if history:
            df = pd.DataFrame(history)
            df["t"] = pd.to_datetime(df["t"], unit="s")
            df.rename(columns={"t": "timestamp", "p": "price"}, inplace=True)
            df["outcome"] = token["outcome"]
            all_dfs.append(df)

    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

df = get_prices_history(ex_cond_id)
print(df)

Empty DataFrame
Columns: []
Index: []
